In [ ]:
import os

import cmcrameri.cm as cmc
import numpy as np
import torch
import yaml
from data import PtychographyDataset
from matplotlib import colors
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
from ptychi_utils import place_patches_fourier_shift

%matplotlib inline
plt.style.use('/home/beams/AILEENLUO/stylesheet.mplstyle')

In [ ]:
results_dir = '/scratch/aileenluo/ptycho-vit/results/run130'

gt_diff_test = np.load(os.path.join(results_dir, 'gt_diff_scan807.npy'))
gt_amp_test = np.load(os.path.join(results_dir, 'gt_amp_scan807.npy'))
gt_ph_test = np.load(os.path.join(results_dir, 'gt_ph_scan807.npy'))
pred_diff_test = np.load(os.path.join(results_dir, 'pred_diff_scan807.npy'))
pred_amp_test = np.load(os.path.join(results_dir, 'pred_amp_scan807.npy'))
pred_ph_test = np.load(os.path.join(results_dir, 'pred_ph_scan807.npy'))

gt_amp_test.shape

In [ ]:
config_path = '/scratch/aileenluo/ptycho-vit/models/run130/config.yaml'

def load_config(path):
    """Load configuration from YAML file."""
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
        return config

config = load_config(config_path)

In [ ]:
# Data for object patch placement
#data_file = '/home/beams/AILEENLUO/ptycho_simulation_factory/outputs/coins256/coins256_dp.hdf5'
#data_file = '/home/beams/AILEENLUO/ptycho_simulation_factory/outputs/cat256/cat_dp.hdf5'
data_file = '/home/beams/AILEENLUO/ptycho_simulation_factory/outputs/scan807/scan807_dp.hdf5'
#data_file = '/scratch/aileenluo/ptycho-vit/data/cameraman256_dp.hdf5'
dataset = PtychographyDataset(
    data_file, 
    config['data']['scale'],
    '/home/beams/AILEENLUO/ptycho_simulation_factory/test2probes_norm.pkl'
    #config['data'].get('normalization_dict_path')
)
tmp = dataset[0] # Just to extract object and position information
positions = dataset._cached_probe_positions
object_size = dataset.object_shape
object_size

In [ ]:
pred_amp_object = torch.zeros(object_size)
pred_ph_object = torch.zeros(object_size)
buffer = torch.zeros(object_size)

pred_amp_object = place_patches_fourier_shift(
    pred_amp_object, 
    positions, 
    torch.from_numpy(pred_amp_test[:, 64:-64, 64:-64]), 
    "add", 
    False, 
    pad=32
    )
pred_ph_object = place_patches_fourier_shift(
    pred_ph_object, 
    positions, 
    torch.from_numpy(pred_ph_test[:, 64:-64, 64:-64]), 
    "add", 
    False, 
    pad=32
    )
buffer = place_patches_fourier_shift(
    buffer, 
    positions, 
    torch.ones_like(torch.from_numpy(pred_ph_test[:, 64:-64, 64:-64])), 
    "add", 
    False, 
    pad=32
    )

gt_amp_object = torch.zeros(object_size)
gt_ph_object = torch.zeros(object_size)

gt_amp_object = place_patches_fourier_shift(
    gt_amp_object, 
    positions, 
    torch.from_numpy(gt_amp_test[:, 64:-64, 64:-64]), 
    "add", 
    False,
    pad=32
    )
gt_ph_object = place_patches_fourier_shift(
    gt_ph_object, 
    positions, 
    torch.from_numpy(gt_ph_test[:, 64:-64, 64:-64]), 
    "add", 
    False, 
    pad=32
    )


In [ ]:
pred_amp_object = torch.zeros(object_size)
pred_ph_object = torch.zeros(object_size)
buffer = torch.zeros(object_size)

pred_amp_object = place_patches_fourier_shift(
    pred_amp_object, 
    positions, 
    torch.from_numpy(pred_amp_test[:, 80:-80, 80:-80]), 
    "add", 
    False, 
    pad=32
    )
pred_ph_object = place_patches_fourier_shift(
    pred_ph_object, 
    positions, 
    torch.from_numpy(pred_ph_test[:, 80:-80, 80:-80]), 
    "add", 
    False, 
    pad=32
    )
buffer = place_patches_fourier_shift(
    buffer, 
    positions, 
    torch.ones_like(torch.from_numpy(pred_ph_test[:, 80:-80, 80:-80])), 
    "add", 
    False, 
    pad=32
    )

gt_amp_object = torch.zeros(object_size)
gt_ph_object = torch.zeros(object_size)

gt_amp_object = place_patches_fourier_shift(
    gt_amp_object, 
    positions, 
    torch.from_numpy(gt_amp_test[:, 80:-80, 80:-80]), 
    "add", 
    False,
    pad=32
    )
gt_ph_object = place_patches_fourier_shift(
    gt_ph_object, 
    positions, 
    torch.from_numpy(gt_ph_test[:, 80:-80, 80:-80]), 
    "add", 
    False, 
    pad=32
    )

In [ ]:
pred_amp_object = pred_amp_object / torch.clip(buffer, min=1)
pred_ph_object = pred_ph_object / torch.clip(buffer, min=1)

gt_amp_object = gt_amp_object / torch.clip(buffer, min=1)
gt_ph_object = gt_ph_object / torch.clip(buffer, min=1)

In [ ]:
plt.imshow(buffer[67:-67, 67:-67], interpolation='none')

In [ ]:
f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2)

gt0 = ax[0, 0].imshow(gt_amp_object[99:-125, 99:-125], interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')

gt1 = ax[0, 1].imshow(gt_ph_object[99:-125, 99:-125], interpolation='none', vmin=-0.5, vmax=0.5, cmap=cmc.vik)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt1, cax=cax, orientation='vertical')
ax[0, 1].set_title('GT phase')

pred0 = ax[1, 0].imshow(pred_amp_object[99:-125, 99:-125], interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred0, cax=cax, orientation='vertical')
ax[1, 0].set_title('Predicted amplitude')

pred1 = ax[1, 1].imshow(pred_ph_object[99:-125, 99:-125], interpolation='none', vmin=-1, vmax=0, cmap=cmc.vik)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred1, cax=cax, orientation='vertical')
ax[1, 1].set_title('Predicted phase')

In [ ]:
f, ax = plt.subplots(figsize=(9, 7), ncols=2, nrows=2)

gt0 = ax[0, 0].imshow(gt_amp_object[99:-99, 99:-99], interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')

gt1 = ax[0, 1].imshow(gt_ph_object[99:-99, 99:-99], interpolation='none', vmin=-0.5, vmax=0.5, cmap=cmc.vik)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt1, cax=cax, orientation='vertical')
ax[0, 1].set_title('GT phase')

pred0 = ax[1, 0].imshow(pred_amp_object[99:-99, 99:-99], interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred0, cax=cax, orientation='vertical')
ax[1, 0].set_title('Predicted amplitude')

pred1 = ax[1, 1].imshow(pred_ph_object[99:-99, 99:-99], interpolation='none', vmin=-1, vmax=0, cmap=cmc.vik)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred1, cax=cax, orientation='vertical')
ax[1, 1].set_title('Predicted phase')

In [ ]:
f, ax = plt.subplots(figsize=(9, 5), ncols=2, nrows=2)

gt0 = ax[0, 0].imshow(gt_amp_object[112:-112, 112:-112], interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')

gt1 = ax[0, 1].imshow(gt_ph_object[112:-112, 112:-112], interpolation='none', vmin=-1, vmax=1, cmap=cmc.vik)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt1, cax=cax, orientation='vertical')
ax[0, 1].set_title('GT phase')

pred0 = ax[1, 0].imshow(pred_amp_object[112:-112, 112:-112], interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred0, cax=cax, orientation='vertical')
ax[1, 0].set_title('Predicted amplitude')

pred1 = ax[1, 1].imshow(pred_ph_object[112:-112, 112:-112], interpolation='none', vmin=-2.2, vmax=-0.2, cmap=cmc.vik)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred1, cax=cax, orientation='vertical')
ax[1, 1].set_title('Predicted phase')

In [ ]:
f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2)

gt0 = ax[0, 0].imshow(gt_amp_object[180:-180, 180:-180], interpolation='none', vmin=0.90, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')

gt1 = ax[0, 1].imshow(gt_ph_object[180:-180, 180:-180], interpolation='none', vmin=-1.3, vmax=1.3, cmap=cmc.vik)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt1, cax=cax, orientation='vertical')
ax[0, 1].set_title('GT phase')

pred0 = ax[1, 0].imshow(pred_amp_object[180:-180, 180:-180], interpolation='none', vmin=0.9, vmax=1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred0, cax=cax, orientation='vertical')
ax[1, 0].set_title('Predicted amplitude')

pred1 = ax[1, 1].imshow(pred_ph_object[180:-180, 180:-180], interpolation='none', vmin=-1.7, vmax=0.3, cmap=cmc.vik)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred1, cax=cax, orientation='vertical')
ax[1, 1].set_title('Predicted phase')

In [ ]:
f, ax = plt.subplots(figsize=(9, 6), ncols=2, nrows=2)

gt0 = ax[0, 0].imshow(gt_amp_object[140:-140, 170:-170], interpolation='none', vmin=0.9, vmax=1.1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')

gt1 = ax[0, 1].imshow(gt_ph_object[140:-140, 170:-170], interpolation='none', cmap=cmc.vik)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt1, cax=cax, orientation='vertical')
ax[0, 1].set_title('GT phase')

pred0 = ax[1, 0].imshow(pred_amp_object[140:-140, 170:-170], interpolation='none', vmin=0.2, vmax=0.8, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred0, cax=cax, orientation='vertical')
ax[1, 0].set_title('Predicted amplitude')

pred1 = ax[1, 1].imshow(pred_ph_object[140:-140, 170:-170], interpolation='none', vmin=-1.5, vmax=0, cmap=cmc.vik)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(pred1, cax=cax, orientation='vertical')
ax[1, 1].set_title('Predicted phase')

In [ ]:
pred_amp_object.min(), pred_amp_object.max(), pred_ph_object.min(), pred_ph_object.max()

In [ ]:
def gen_plot(gt_diff, gt_amp, gt_ph, pred_diff, pred_amp, pred_ph):
    f, ax = plt.subplots(figsize=(7, 4), nrows=2, ncols=3)

    in0 = ax[0, 0].imshow(gt_diff, interpolation='none', norm=colors.LogNorm(vmin=0.1, vmax=np.sqrt(10000)), cmap=cmc.lapaz)
    divider = make_axes_locatable(ax[0, 0])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(in0, cax=cax, orientation='vertical')
    ax[0, 0].set_title('Input Diff. Amp.')

    in1 = ax[0, 1].imshow(gt_amp, interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
    divider = make_axes_locatable(ax[0, 1])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(in1, cax=cax, orientation='vertical', format='%.2f')
    ax[0, 1].set_title('GT Amp.')

    in2 = ax[0, 2].imshow(gt_ph, interpolation='none', vmin=-1, vmax=1, cmap=cmc.vik)
    divider = make_axes_locatable(ax[0, 2])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(in2, cax=cax, orientation='vertical', format='%.1f')
    ax[0, 2].set_title('GT Phase')

    out0 = ax[1, 0].imshow(pred_diff, interpolation='none', norm=colors.LogNorm(vmin=0.1, vmax=np.sqrt(10000)), 
                           cmap=cmc.lapaz)
    divider = make_axes_locatable(ax[1, 0])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(out0, cax=cax, orientation='vertical')
    ax[1, 0].set_title('Output Diff. Amp.')

    out1 = ax[1, 1].imshow(pred_amp, interpolation='none', vmin=0.95, vmax=1, cmap=cmc.oslo)
    divider = make_axes_locatable(ax[1, 1])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(out1, cax=cax, orientation='vertical', format='%.2f')
    ax[1, 1].set_title('Predicted Amp.')

    out2 = ax[1, 2].imshow(pred_ph, interpolation='none', cmap=cmc.vik)
    divider = make_axes_locatable(ax[1, 2])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    f.colorbar(out2, cax=cax, orientation='vertical', format='%.1f')
    ax[1, 2].set_title('Predicted Phase')

    plt.tight_layout()
    return f, ax

In [ ]:
f, ax = gen_plot(gt_diff_test[5000], gt_amp_test[5000], gt_ph_test[5000], 
                 pred_diff_test[5000], pred_amp_test[5000], pred_ph_test[5000])